<a href="https://colab.research.google.com/github/landpd/Generador-Materiales-Pulppo/blob/main/Generacion_de_piezas_creativas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# CELDA A: INSTALAR MOTOR DE DISEÑO
# ==========================================

!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update -y
!apt-get install -y google-chrome-stable
!pip install html2image
print("✅ ¡Motor de Google Chrome instalado correctamente!")

OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 https://cli.github.com/packages stable InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,207 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:14 http://security

In [3]:
# ==========================================
# CELDA B: GENERADOR DE DISEÑOS MODULAR
# ==========================================

import os
import glob
import re
import base64
import requests
import pandas as pd
import shutil
from html2image import Html2Image
from google.colab import drive

# ==============================================================================
# --- 1. ZONA DE PLANTILLAS ---
# ==============================================================================

def disenio_vertical_3fotos(datos):
    """ Plantilla Clásica: 1080 x 1350 con 3 fotos """
    return f"""
    <html>
    <head>
    <link href="https://fonts.googleapis.com/css2?family=EB+Garamond:wght@500&family=Nunito+Sans:wght@400;900&display=swap" rel="stylesheet">
    <style>
        body {{ margin: 0; padding: 0; width: 1080px; height: 1350px; background: #000; overflow: hidden; }}
        .container {{ display: flex; flex-direction: column; width: 1080px; height: 1350px; }}

        .img-box {{ width: 1080px; height: 450px; position: relative; overflow: hidden; }}
        .img-box img {{ width: 100%; height: 100%; object-fit: cover; }}

        .overlay {{
            position: absolute; top: 0; left: 0; width: 100%; height: 100%;
            background-color: rgba(33, 35, 34, 0.3);
            display: flex; flex-direction: column; justify-content: space-between;
            padding: 40px 60px; box-sizing: border-box;
        }}

        .block-top {{ text-align: right; width: 100%; }}
        .block-middle {{ text-align: left; width: 100%; }}
        .block-bottom {{ text-align: right; width: 100%; }}

        .colonia {{ font-family: 'Nunito Sans', sans-serif; font-weight: 400; font-size: 18pt; text-transform: uppercase; color: #FAFAFA; margin-bottom: 5px; }}
        .calle {{ font-family: 'EB Garamond', serif; font-weight: 500; font-size: 42pt; color: #FAFAFA; margin: 0; line-height: 1.1; }}

        .tipo {{ font-family: 'Nunito Sans', sans-serif; font-weight: 900; font-size: 16pt; text-transform: uppercase; color: #FAFAFA; margin-bottom: 5px; }}
        .precio {{ font-family: 'Nunito Sans', sans-serif; font-weight: 900; font-size: 16pt; text-transform: uppercase; color: #FAFAFA; margin-bottom: 0; }}

        .atributos {{ font-family: 'Nunito Sans', sans-serif; font-weight: 400; font-size: 16pt; text-transform: uppercase; color: #FAFAFA; line-height: 1.3; }}

        .text-shadow {{ text-shadow: 1px 1px 4px rgba(0,0,0,0.4); }}
    </style>
    </head>
    <body>
        <div class="container">
            <div class="img-box"><img src="{datos['img1']}"></div>
            <div class="img-box">
                <img src="{datos['img2']}">
                <div class="overlay">
                    <div class="block-top text-shadow">
                        <div class="tipo">{datos['tipo_operacion']}</div>
                        <div class="precio">{datos['precio']}</div>
                    </div>
                    <div class="block-middle text-shadow">
                        <div class="colonia">{datos['colonia_estado']}</div>
                        <h1 class="calle">{datos['calle']}</h1>
                    </div>
                    <div class="block-bottom text-shadow">
                        <div class="atributos">
                            {datos['atributos_html']}
                        </div>
                    </div>
                </div>
            </div>
            <div class="img-box"><img src="{datos['img3']}"></div>
        </div>
    </body>
    </html>
    """

def disenio_landscape_5fotos(datos):
    """ Nueva Plantilla Landscape: 1920 x 1080 con 5 fotos """
    return f"""
    <html>
    <head>
    <link href="https://fonts.googleapis.com/css2?family=EB+Garamond:wght@400;500&family=Nunito+Sans:wght@400;700;900&display=swap" rel="stylesheet">
    <style>
        body {{ margin: 0; padding: 0; width: 1920px; height: 1080px; background: #FAFAFA; overflow: hidden; }}
        .container {{ position: relative; width: 1920px; height: 1080px; box-sizing: border-box; }}

        /* --- FOTOS --- */
        .img-box {{ position: absolute; overflow: hidden; background: #eaeaea; }}
        .img-box img {{ width: 100%; height: 100%; object-fit: cover; }}

        .img1 {{ left: 40px; top: 40px; width: 1100px; height: 700px; }}
        .img2 {{ right: 40px; top: 110px; width: 700px; height: 450px; }}
        .img3 {{ left: 1180px; top: 590px; width: 260px; height: 300px; }}
        .img4 {{ right: 40px; top: 590px; width: 410px; height: 210px; }}
        .img5 {{ right: 40px; top: 830px; width: 410px; height: 210px; }}

        /* --- ETIQUETA SUPERIOR --- */
        .tipo-etiqueta {{
            position: absolute; right: 40px; top: 50px;
            font-family: 'Nunito Sans', sans-serif; font-weight: 900; font-size: 26px;
            text-transform: uppercase; color: #212322; letter-spacing: 2px;
        }}

        /* --- ZONA INFERIOR (Textos y Atributos agrupados para evitar solapamiento) --- */
        .bottom-zone {{
            position: absolute; left: 40px; top: 760px;
            width: 1100px; /* Delimita el ancho para que no toque las fotos chicas */
            height: 280px;
            display: flex; align-items: center; gap: 50px;
        }}

        .text-container {{
            flex: 1; /* Toma todo el espacio disponible empujando los atributos a la derecha */
            display: flex; flex-direction: column; justify-content: center;
        }}
        .calle {{ font-family: 'EB Garamond', serif; font-weight: 500; font-size: 60px; margin: 0; line-height: 1.1; color: #212322; }}
        .colonia {{ font-family: 'EB Garamond', serif; font-weight: 400; font-size: 40px; margin: 0; line-height: 1.2; padding-top: 10px; color: #212322; }}
        .divider {{ width: 100%; height: 2px; background-color: #D1D1D1; margin-top: 25px; }}

        /* --- ATRIBUTOS (Grid 2x2) --- */
        .atributos {{
            flex-shrink: 0; /* Impide que los atributos se encojan, forzando a la calle a crear saltos de línea si es muy larga */
            display: grid;
            grid-template-columns: max-content max-content; /* Fuerza exactamente 2 columnas */
            gap: 15px 40px; /* gap de filas y columnas */
        }}
        .atributos div {{
            font-family: 'Nunito Sans', sans-serif; font-weight: 500; font-size: 24px;
            text-transform: uppercase; color: #212322;
            white-space: nowrap; /* Evita que el texto de un atributo se divida en dos líneas */
        }}
        .atributos div::before {{
            content: '/ '; color: #F6BE00; font-weight: 900;
        }}
    </style>
    </head>
    <body>
        <div class="container">
            <div class="tipo-etiqueta">[ {datos['tipo_operacion']} ]</div>

            <div class="img-box img1"><img src="{datos['img1']}"></div>
            <div class="img-box img2"><img src="{datos['img2']}"></div>
            <div class="img-box img3"><img src="{datos['img3']}"></div>
            <div class="img-box img4"><img src="{datos['img4']}"></div>
            <div class="img-box img5"><img src="{datos['img5']}"></div>

            <!-- Bloque Inferior agrupado -->
            <div class="bottom-zone">
                <div class="text-container">
                    <h1 class="calle">{datos['calle']}</h1>
                    <h2 class="colonia">{datos['colonia_estado']}</h2>
                    <div class="divider"></div>
                </div>
                <div class="atributos">
                    {datos['atributos_html']}
                </div>
            </div>
        </div>
    </body>
    </html>
    """

# ==============================================================================
# --- 2. PLANTILLAS A GENERAR: ---
# ==============================================================================
PLANTILLAS_A_GENERAR = {
    "Post_3Fotos": {
        "funcion": disenio_vertical_3fotos,
        "width": 1080,
        "height": 1350,
        "sufijo_archivo": "post_3fotos"
    },
    "Landscape_5Fotos": {
        "funcion": disenio_landscape_5fotos,
        "width": 1920,
        "height": 1080,
        "sufijo_archivo": "landscape_5fotos"
    }
}

# ==============================================================================
# --- 3. CONFIGURACIÓN DEL ENTORNO Y DATOS ---
# ==============================================================================
drive.mount('/content/drive')

RUTA_CSV = "/content/drive/MyDrive/Nuevos proyectos/CSV Metabase"
RUTA_SALIDA = "/content/drive/MyDrive/Nuevos proyectos/Diseños_Generados"
os.makedirs(RUTA_SALIDA, exist_ok=True)

hti = Html2Image(
    browser_executable='/usr/bin/google-chrome',
    custom_flags=['--no-sandbox', '--disable-gpu', '--hide-scrollbars', '--disable-dev-shm-usage']
)
hti.output_path = '/content'

csv_files = glob.glob(os.path.join(RUTA_CSV, "propiedades_1_5_10_con_fotos_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No se encontró ningún archivo CSV en: {RUTA_CSV}")

latest_csv = sorted(csv_files)[-1]
print(f"📄 Procesando base de datos: {os.path.basename(latest_csv)}")
df = pd.read_csv(latest_csv)

# --- Funciones de ayuda ---
def formatear_atributo(valor, sufijo):
    val_str = str(valor).strip()
    if val_str.lower() in ['nan', 'none', '', '0', '0.0']: return ""
    if val_str.endswith('.0'): val_str = val_str[:-2]
    return f"{val_str} {sufijo}"

def url_to_base64(url):
    if not url: return ""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        encoded = base64.b64encode(resp.content).decode('utf-8')
        return f"data:image/jpeg;base64,{encoded}"
    except:
        return ""

# ==============================================================================
# --- 4. BUCLE PRINCIPAL DE PROCESAMIENTO Y RENDERIZADO ---
# ==============================================================================
print("\n🚀 Iniciando generación de imágenes modulares...\n")
imagenes_generadas = 0

for index, row in df.iterrows():
    internal_id = str(row['InternalId']).strip()
    if internal_id == 'nan': continue

    print(f"\nPropiedad: {internal_id}...")

    # --- LIMPIEZA DE DATOS ---
    company_raw = str(row.get('Company: Name', 'Inmobiliaria')).strip()
    if company_raw.lower() == 'nan': company_raw = "Inmobiliaria"
    company_clean = re.sub(r'[^A-Za-z0-9]+', '_', company_raw).strip('_')

    # --- FOTOS ---
    pictures_raw = str(row['Pictures'])
    urls = re.findall(r'\[:url\s+"([^"]+)"\]', pictures_raw)
    if not urls: continue

    def obtener_foto(indice):
        return urls[indice] if len(urls) > indice else urls[0]

    img1_b64 = url_to_base64(obtener_foto(0))
    img2_b64 = url_to_base64(obtener_foto(1))
    img3_b64 = url_to_base64(obtener_foto(2))
    img4_b64 = url_to_base64(obtener_foto(3))
    img5_b64 = url_to_base64(obtener_foto(4))

    # --- TEXTOS ---
    calle_cruda = str(row.get('Address: Street', '')).strip()
    if calle_cruda.lower() == 'nan':
        calle = ""
    else:
        calle = calle_cruda.split(',')[0].strip()
    colonia = str(row.get('Address: Neighborhood: Name', '')).strip()
    if colonia.lower() == 'nan': colonia = ""
    estado = str(row.get('Address: State: Name', '')).strip()
    if estado.lower() == 'nan': estado = ""
    colonia_estado = f"{colonia}, {estado}".strip(", ")

    tipo = str(row.get('Type', '')).strip()
    if tipo.lower() == 'nan': tipo = ""
    operacion = str(row.get('Listing: Operation', '')).strip()
    if operacion.lower() == 'nan': operacion = ""
    if operacion.lower() == 'sale': operacion = 'venta'
    elif operacion.lower() == 'rent': operacion = 'renta'

    if tipo and operacion: tipo_en_operacion = f"{tipo} en {operacion.lower()}"
    elif tipo: tipo_en_operacion = tipo
    elif operacion: tipo_en_operacion = operacion.capitalize()
    else: tipo_en_operacion = ""

    precio_crudo = str(row.get('Listing: Price: Price', '0')).strip()
    moneda_crudo = str(row.get('Listing: Price: Currency', 'mxn')).strip().lower()
    if moneda_crudo == 'nan' or moneda_crudo == '': moneda_crudo = "mxn"
    precio_limpio = precio_crudo.replace(',', '').replace('$', '').replace(' ', '')
    try:
        precio_formateado = f"${float(precio_limpio):,.2f} {moneda_crudo}"
    except:
        precio_formateado = precio_crudo if precio_crudo.lower() != 'nan' else ""

        attr_list = []
    for val, sufijo in [(row.get('Attributes: Suites', ''), 'habitaciones'),
                        (row.get('Attributes: Bathrooms', ''), 'baños'),
                        (row.get('Attributes: Parkings', ''), 'estacionamientos'),
                        (row.get('Attributes: TotalSurface', ''), 'm² TOTALES')]: # <-- CAMBIO APLICADO AQUÍ
        attr = formatear_atributo(val, sufijo)
        if attr: attr_list.append(attr)
    atributos_html = "".join([f"<div>{a}</div>" for a in attr_list])

    # --- EMPAQUETADO DE DATOS ---
    datos_propiedad = {
        "img1": img1_b64,
        "img2": img2_b64,
        "img3": img3_b64,
        "img4": img4_b64,
        "img5": img5_b64,
        "tipo_operacion": tipo_en_operacion,
        "precio": precio_formateado,
        "colonia_estado": colonia_estado,
        "calle": calle,
        "atributos_html": atributos_html
    }

    # --- RENDERIZADO DINÁMICO DE PLANTILLAS ---
    for nombre_disenio, config in PLANTILLAS_A_GENERAR.items():
        print(f"  -> Verificando {nombre_disenio}...")

        nombre_archivo = f"{internal_id}_{company_clean}_{config['sufijo_archivo']}.jpg"
        ruta_guardado = os.path.join(RUTA_SALIDA, nombre_archivo)
        ruta_local = os.path.join('/content', nombre_archivo)

        # VALIDACIÓN: Omitir si ya existe
        if os.path.exists(ruta_guardado):
            print(f"     ✅ El diseño {nombre_archivo} ya existe. Omitiendo...")
            continue

        print(f"  -> 🎨 Diseñando {nombre_disenio}...")

        html_final = config["funcion"](datos_propiedad)

        hti.screenshot(
            html_str=html_final,
            save_as=nombre_archivo,
            size=(config["width"], config["height"])
        )

        shutil.move(ruta_local, ruta_guardado)
        imagenes_generadas += 1

print(f"\n✅ ¡Proceso terminado! Se generaron {imagenes_generadas} diseños en total.")
print(f"📂 Revisa la carpeta: {RUTA_SALIDA}")

Mounted at /content/drive
📄 Procesando base de datos: propiedades_1_5_10_con_fotos_2026-06-04T16_50_18.259815671-06_00.csv

🚀 Iniciando generación de imágenes modulares...


Propiedad: BCW-028...
  -> Verificando Post_3Fotos...
     ✅ El diseño BCW-028_Ponton_Asesores_post_3fotos.jpg ya existe. Omitiendo...
  -> Verificando Landscape_5Fotos...
     ✅ El diseño BCW-028_Ponton_Asesores_landscape_5fotos.jpg ya existe. Omitiendo...

Propiedad: CAN-505...
  -> Verificando Post_3Fotos...
     ✅ El diseño CAN-505_Habitanza_post_3fotos.jpg ya existe. Omitiendo...
  -> Verificando Landscape_5Fotos...
     ✅ El diseño CAN-505_Habitanza_landscape_5fotos.jpg ya existe. Omitiendo...

Propiedad: CAQ-399...
  -> Verificando Post_3Fotos...
     ✅ El diseño CAQ-399_Habitanza_post_3fotos.jpg ya existe. Omitiendo...
  -> Verificando Landscape_5Fotos...
     ✅ El diseño CAQ-399_Habitanza_landscape_5fotos.jpg ya existe. Omitiendo...

Propiedad: CAZ-272...
  -> Verificando Post_3Fotos...
     ✅ El diseño CA